<a href="https://colab.research.google.com/github/Archee-Khandelwal/isro-react-agent/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

**Modified by Archee Khandelwal.** The agent answers questions about Indian space missions (ISRO) by reasoning, calling a `search_documents` tool over a small document set, and answering only from what it finds.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [18]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [19]:
# Quiet ensures your UI is not full of logs.
# google-genai is the SDK the agent loop below actually uses.
!pip install -q -U google-genai
print("Done.")

Done.


In [20]:
# @title User input needed.
# Documents: Indian space missions (ISRO). Each has 4+ sentences.
DOCUMENTS = {
    "chandrayaan_3": (
        "Chandrayaan-3 is India's third lunar exploration mission, developed by the Indian Space Research Organisation (ISRO). "
        "It was launched on 14 July 2023 on the LVM3 rocket from the Satish Dhawan Space Centre in Sriharikota. "
        "Its lander Vikram touched down near the Moon's south polar region on 23 August 2023, making India the first country to land near the lunar south pole. "
        "The mission also carried a small rover named Pragyan, which drove on the lunar surface and studied the soil. "
        "23 August is now celebrated in India as National Space Day."
    ),
    "mangalyaan": (
        "The Mars Orbiter Mission, popularly called Mangalyaan, was India's first interplanetary mission. "
        "It was launched on 5 November 2013 on a PSLV rocket from Sriharikota. "
        "The spacecraft entered orbit around Mars on 24 September 2014. "
        "India became the first Asian nation to reach Mars orbit and the first country to do so on its first attempt. "
        "The mission was famous for its very low cost compared with other Mars missions."
    ),
    "aditya_l1": (
        "Aditya-L1 is India's first space-based mission to study the Sun. "
        "It was launched on 2 September 2023 on a PSLV rocket. "
        "The spacecraft was placed in a halo orbit around the Sun-Earth Lagrange point L1, about 1.5 million kilometres from Earth, on 6 January 2024. "
        "From L1 it can watch the Sun continuously without being blocked by Earth or eclipses. "
        "It carries seven instruments to study the solar corona, solar wind and solar flares."
    ),
    "pslv_c37": (
        "PSLV stands for Polar Satellite Launch Vehicle and is one of ISRO's most reliable rockets. "
        "On 15 February 2017, the PSLV-C37 mission launched 104 satellites in a single flight. "
        "This was a world record at the time for the most satellites launched on one rocket. "
        "The main payload was the Cartosat-2D Earth observation satellite, and most of the other satellites were small nanosatellites from other countries. "
        "The record showed ISRO's ability to offer low-cost launch services to international customers."
    ),
}

assert len(DOCUMENTS) >= 4, "Need at least 4 documents"
print(f"{len(DOCUMENTS)} documents loaded:", list(DOCUMENTS))

4 documents loaded: ['chandrayaan_3', 'mangalyaan', 'aditya_l1', 'pslv_c37']


In [21]:
# @title Define a tool
# Common words that would otherwise match every document.
STOPWORDS = {"a","an","the","is","was","of","in","on","to","and","or","for","what","when",
             "which","who","how","did","does","do","it","its","by","with","at","from","as","many"}

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []
    words = {w.strip("?.,!") for w in query_lower.split()} - STOPWORDS

    for doc_name, content in DOCUMENTS.items():
        match_count = sum(1 for word in words if word and word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check (was search_recipes in the starter code -> NameError).
print(search_documents("Mars orbit"))

[Source: mangalyaan]
The Mars Orbiter Mission, popularly called Mangalyaan, was India's first interplanetary mission. It was launched on 5 November 2013 on a PSLV rocket from Sriharikota. The spacecraft entered orbit around Mars on 24 September 2014. India became the first Asian nation to reach Mars orbit and the first country to do so on its first attempt. The mission was famous for its very low cost compared with other Mars missions.

[Source: aditya_l1]
Aditya-L1 is India's first space-based mission to study the Sun. It was launched on 2 September 2023 on a PSLV rocket. The spacecraft was placed in a halo orbit around the Sun-Earth Lagrange point L1, about 1.5 million kilometres from Earth, on 6 January 2024. From L1 it can watch the Sun continuously without being blocked by Earth or eclipses. It carries seven instruments to study the solar corona, solar wind and solar flares.


In [22]:
# @title Choose the model and system instruction
# Fix: "claude" is not a Gemini model. Using a current Gemini model from the developer docs:
# https://ai.google.dev/gemini-api/docs/models
MODEL_NAME = "gemini-3.5-flash-lite"   # if this errors, try "gemini-3.6-flash"

SYSTEM_INSTRUCTION = """
You are a space science research assistant specialising in Indian space missions (ISRO).
When you need specific facts such as launch dates, rockets, records or mission goals,
use the search_documents tool. You may call it more than once with different queries.
Only answer based on what you find in the documents.
If the documents do not contain the answer, say so clearly.
"""

print("Model ready:", MODEL_NAME)

Model ready: gemini-3.5-flash-lite


In [17]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the ISRO mission documents and return the most relevant paragraphs.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,  # fix: was never passed to the model
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)  # fix: was search_recipes
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("Which mission made India the first country to land near the Moon's south pole?")

Question: Which mission made India the first country to land near the Moon's south pole?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Moon south pole first country land mission'})
  Result preview: [Source: chandrayaan_3]
Chandrayaan-3 is India's third lunar exploration mission, developed by the I...

Final answer:
**Chandrayaan-3** made India the first country to land near the Moon's south pole. Its lander, Vikram, successfully touched down in the lunar south polar region on August 23, 2023.


In [15]:
# @title Ask follow up questions
questions = [
    "When did Mangalyaan reach Mars, and why was it a record?",
    "How far from Earth is Aditya-L1 and what does it study?",
    "How many satellites did PSLV-C37 launch and what was the main payload?",
    # This question is not answered by the database.
    "How many ghosts eat carrots everyday?",
    # This question is part of the general knowledge of the model.
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    run_agent(q)


Question: When did Mangalyaan reach Mars, and why was it a record?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Mangalyaan reach Mars record date'})
  Result preview: [Source: mangalyaan]
The Mars Orbiter Mission, popularly called Mangalyaan, was India's first interp...

Final answer:
Based on the ISRO mission documents, Mangalyaan reached Mars (entered orbit around Mars) on **24 September 2014**.

It was a record because:
1. India became the **first Asian nation** to reach Mars orbit.
2. India became the **first country** to do so on its **first attempt**.

Question: How far from Earth is Aditya-L1 and what does it study?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'Aditya-L1 distance from Earth orbit Lagrange point 1 study science objectives'})
  Result preview: [Source: aditya_l1]
Aditya-L1 is India's first space-based mission to study the Sun. It was launched...

Final answe

In [16]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)